# Taller 2: Expedición Lunar — Cartas desde la Luna con LLMs

## Bienvenidos, astronautas

El año es **2042**. Ustedes forman parte de la tripulación **Artemis VII**, la primera misión de larga duración en la superficie lunar. Llevan 3 meses en la **Base Selene** y hoy, por fin, la ventana de comunicación con la Tierra se ha abierto.

Tienen una única oportunidad para enviar una carta a sus familias. Pero aquí está el problema: el sistema de comunicaciones de la base usa un **modelo de lenguaje (LLM)** como asistente de redacción. Dependiendo de cómo configuren sus parámetros, la carta será:

- Fría y precisa... o cálida y creativa
- Corta y directa... o larga y detallada
- Repetitiva y monótona... o variada y expresiva

Su misión en este taller: **dominar los parámetros del LLM para escribir la carta perfecta.**

### Lo que aprenderán hoy

| Parámetro | Qué controla | Equivalente en Ollama |
|---|---|---|
| `temperature` | Creatividad vs precisión | `temperature` |
| `top_p` | Diversidad del vocabulario | `top_p` |
| `max_tokens` | Longitud de la respuesta | `num_predict` |
| `frequency_penalty` | Penalización por repetición | `repeat_penalty` |

**Tiempo estimado:** 30 minutos

## 1. Preparación de la Base Lunar (Instalación)

Antes de conectar con el sistema de comunicaciones, necesitamos instalar las herramientas.

**Requisitos previos:**
- Tener [Ollama](https://ollama.com/) instalado y corriendo (`ollama serve` en una terminal)
- Haber descargado el modelo: `ollama pull gemma:2b`

In [4]:
# Instalamos la libreria de Python para comunicarnos con Ollama
!pip install ollama -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import ollama

def generar_carta(prompt, opciones=None, modelo="gemma2:2b"):
    """
    Genera texto usando Ollama con las opciones especificadas.

    Args:
        prompt: El texto de instruccion para el modelo.
        opciones: Diccionario con parametros (temperature, top_p, num_predict, repeat_penalty).
        modelo: Nombre del modelo a usar.
    """
    respuesta = ollama.generate(
        model=modelo,
        prompt=prompt,
        options=opciones or {},
        think=False  # Desactiva el razonamiento interno (thinking tokens) que consume num_predict
    )
    print("=" * 60)
    print(f"Parametros usados: {opciones or 'default'}")
    print("=" * 60)
    print(respuesta["response"])
    print(f"\n[Tokens generados: {respuesta.get('eval_count', 'N/A')}]")
    return respuesta["response"]

print("Sistemas de comunicacion listos. Conexion con Ollama establecida.")

Sistemas de comunicacion listos. Conexion con Ollama establecida.


## 2. Primera prueba de comunicación

Antes de ajustar parámetros, probemos que la conexión con el modelo funciona. Enviaremos un mensaje simple.

In [7]:
# Prueba basica de conexion
respuesta = ollama.generate(
    model="gemma2:2b",
    prompt="Como regla solo debes decir:'Conexion establecida desde la Base Selene' y nada mas.",
    think=False
)
print(respuesta["response"])

Conexion establecida desde la Base Selene 



## 3. Temperature: El termómetro emocional

La **temperature** controla qué tan "creativo" o "predecible" es el modelo.

| Valor | Efecto |
|---|---|
| `0.0 - 0.3` | Muy predecible, formal, repetitivo. Como un informe técnico. |
| `0.4 - 0.7` | Equilibrado. Coherente pero con cierta variedad. |
| `0.8 - 1.5` | Muy creativo, impredecible. Puede inventar cosas inesperadas. |

**Escenario:** Van a dictarle al sistema una carta para su familia. Primero con temperature baja (modo "informe de misión") y luego con temperature alta (modo "carta emocional").

In [8]:
prompt_base = """Eres un astronauta en la Luna. Escribe un parrafo corto (3-4 oraciones)
de una carta para tu familia contandoles como fue tu primer paseo lunar."""

# Temperature BAJA: modo informe tecnico
print("TEMPERATURE = 0.1 (Fria, precisa, predecible)")
print()
carta_fria = generar_carta(prompt_base, opciones={"temperature": 0.1})

TEMPERATURE = 0.1 (Fria, precisa, predecible)

Parametros usados: {'temperature': 0.1}
Hola a todos!  La luna es increíblemente hermosa, ¡es como si el cielo se hubiera hecho de piedra!  El paisaje lunar es tan diferente a lo que imaginábamos, con sus montañas y cráteres.  Me siento muy orgulloso de haber sido elegido para esta misión, y estoy ansioso por ver qué más descubrimos en la superficie de la luna. 


[Tokens generados: 79]


In [9]:
# Temperature ALTA: modo carta emotiva
print("TEMPERATURE = 1.2 (Calida, creativa, impredecible)")
print()
carta_calida = generar_carta(prompt_base, opciones={"temperature": 1.2})

TEMPERATURE = 1.2 (Calida, creativa, impredecible)

Parametros usados: {'temperature': 1.2}
Escribo esta carta con las estrellas a mi alrededor y la quietud del silencio lunar en mi mente. ¡Lo que más me impactó de este paseo es la sensación de libertad! Sentí la gravedad lunar sobre mis pies, y aunque su intensidad era diferente, pude explorar con una sensación de maravilla. Nunca había sentido una conexión tan profunda con el universo hasta ahora, un simple paseo lunar que deja un legado inolvidable en mi corazón.  Te mando un abrazo desde la luna. 


[Tokens generados: 101]


## 4. Top_p: El filtro de vocabulario

**Top_p** (nucleus sampling) controla la diversidad del vocabulario que el modelo puede usar.

- `top_p = 0.1` --> Solo usa las palabras más probables. Vocabulario limitado.
- `top_p = 0.9` --> Considera un rango amplio de palabras. Vocabulario rico.

Piensen en esto: cuando escriben un informe técnico usan pocas palabras (top_p bajo). Cuando escriben poesía, usan todo su vocabulario (top_p alto).

**Escenario:** Van a describir lo que ven desde la ventana de la base lunar.

In [10]:
prompt_ventana = """Eres un astronauta en una base lunar. Describe en un parrafo (3-4 oraciones)
lo que ves al mirar por la ventana de la base hacia el espacio."""

# Top_p BAJO: vocabulario restringido
print("TOP_P = 0.1 (Vocabulario restringido)")
print()
vista_simple = generar_carta(prompt_ventana, opciones={"top_p": 0.1, "temperature": 0.7})

print("\n" + "~" * 60 + "\n")

# Top_p ALTO: vocabulario amplio
print("TOP_P = 0.95 (Vocabulario amplio)")
print()
vista_rica = generar_carta(prompt_ventana, opciones={"top_p": 0.95, "temperature": 0.7})

TOP_P = 0.1 (Vocabulario restringido)

Parametros usados: {'top_p': 0.1, 'temperature': 0.7}
Al mirar a través de la ventana de la base lunar, mi vista se adentra en un lienzo negro salpicado de estrellas. Las luces de las ciudades terrestres se asemejan a diminutas luciérnagas en una distancia infinita, mientras que los planetas, con sus colores y formas distintivas, se despliegan como un caleidoscopio cósmico. La inmensidad del espacio me envuelve con una sensación de insignificancia y asombro al mismo tiempo, recordándome la magnitud del universo y nuestra pequeña presencia dentro de él. 


[Tokens generados: 113]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

TOP_P = 0.95 (Vocabulario amplio)

Parametros usados: {'top_p': 0.95, 'temperature': 0.7}
Al mirar a través del cristal de la base lunar, mi vista se abre a un lienzo infinito de estrellas. Las luces tenues de las ciudades terrestres se han desvanecido en la oscuridad profunda del espacio, dejando tras de sí un

## 5. Num_predict (max_tokens): El límite de palabras

**num_predict** controla la longitud máxima de la respuesta (en tokens, donde 1 token es aprox. 3/4 de una palabra).

- `num_predict = 30` --> Respuesta muy corta. Como un telegrama.
- `num_predict = 200` --> Respuesta mediana. Una carta breve.
- `num_predict = 500` --> Respuesta larga. Una carta detallada.

**Escenario:** La ventana de comunicación se está cerrando. Primero solo tienen tiempo para un mensaje de emergencia (corto). Luego la ventana se reabre y pueden enviar un mensaje completo.

In [11]:
prompt_mensaje = """Eres un astronauta en la Luna. Escribe un mensaje para tu familia
explicandoles que estas bien y contandoles algo interesante del dia."""

# Mensaje CORTO: telegrama espacial
print("NUM_PREDICT = 30 (Telegrama espacial)")
print()
telegrama = generar_carta(prompt_mensaje, opciones={"num_predict": 30, "temperature": 0.7})
print("\n" + "~" * 60 + "\n")

# Mensaje LARGO: carta completa
print("NUM_PREDICT = 200 (Carta completa)")
print()
carta = generar_carta(prompt_mensaje, opciones={"num_predict": 200, "temperature": 0.7})

NUM_PREDICT = 30 (Telegrama espacial)

Parametros usados: {'num_predict': 30, 'temperature': 0.7}
¡Hola familia! 🌎🚀

Espero que esto les llegue bien. Estoy aquí, en la Luna, ¡y todo va genial!  El

[Tokens generados: 30]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

NUM_PREDICT = 200 (Carta completa)

Parametros usados: {'num_predict': 200, 'temperature': 0.7}
¡Hola a todos!

Escribir esta nota me llena de nostalgia, ya que estoy aquí, en la Luna, y es increíble. El paisaje lunar es simplemente impresionante. 🌕  El contraste entre el gris intenso de la roca y las sombras del Sol es realmente mágico. 🌌

He aprendido algo curioso hoy: se cree que las primeras misiones espaciales a la Luna fueron guiadas por un fenómeno llamado "**lunar libration**".  Es como si la Luna se movía ligeramente en su órbita alrededor de la Tierra, creando una especie de danza cósmica. ¡Me ha parecido fascinante! 

Estoy muy bien y todo está funcionando perfectamente. La comida es deliciosa (e

## 6. Repeat_penalty: El detector de repetición

**repeat_penalty** penaliza al modelo cuando repite las mismas palabras o frases.

- `repeat_penalty = 1.0` --> Sin penalización. El modelo puede repetir libremente.
- `repeat_penalty = 1.3` --> Penalización moderada. Evita repeticiones obvias.
- `repeat_penalty = 1.8` --> Penalización fuerte. Fuerza mucha variedad (puede volverse incoherente).

**Escenario:** Un compañero de tripulación le pide al sistema que escriba un reporte del día a día en la base. Sin penalización, el modelo tiende a repetirse. Veamos la diferencia.

In [12]:
prompt_reporte = """Eres un astronauta en la Luna. Escribe un reporte de 5 oraciones
describiendo tu rutina diaria en la base lunar. Incluye detalles sobre comida,
experimentos y tiempo libre."""

# Sin penalizacion por repeticion
print("REPEAT_PENALTY = 1.0 (Sin penalizacion)")
print()
reporte_repetitivo = generar_carta(prompt_reporte, opciones={
    "repeat_penalty": 1.0,
    "temperature": 0.7,
    "num_predict": 200
})

print("\n" + "~" * 60 + "\n")

# Con penalizacion moderada
print("REPEAT_PENALTY = 1.5 (Penalizacion moderada)")
print()
reporte_variado = generar_carta(prompt_reporte, opciones={
    "repeat_penalty": 1.5,
    "temperature": 0.7,
    "num_predict": 200
})

REPEAT_PENALTY = 1.0 (Sin penalizacion)

Parametros usados: {'repeat_penalty': 1.0, 'temperature': 0.7, 'num_predict': 200}
Mi día comienza con la rutina de las mañanas, que consiste en revisar los informes de las últimas misiones y preparar los equipos para las tareas del día. Me dedico a supervisar los experimentos de energía solar y la recolección de datos de la actividad lunar.  El almuerzo es un plato de comida nutritiva, con opciones variadas como pasta con salsa de vegetales o un sandwich de pollo con pan integral. En el tiempo libre, disfruto de las vistas de la Luna y leo un libro de ciencia ficción.  Al final del día, me siento satisfecho por haber completado mis tareas y por haber contribuido a la investigación lunar. 


[Tokens generados: 133]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

REPEAT_PENALTY = 1.5 (Penalizacion moderada)

Parametros usados: {'repeat_penalty': 1.5, 'temperature': 0.7, 'num_predict': 200}
Mi día comienza con una breve revisión del

## 7. Combinando todo: La carta perfecta

Ahora que conocen cada parámetro, es hora de combinarlos. Aquí hay dos "perfiles" predefinidos:

| Perfil | temperature | top_p | num_predict | repeat_penalty | Uso ideal |
|---|---|---|---|---|---|
| **Informe técnico** | 0.2 | 0.3 | 150 | 1.0 | Reportes formales |
| **Carta personal** | 0.8 | 0.9 | 250 | 1.3 | Mensajes emotivos |

In [13]:
prompt_carta_final = """Eres un astronauta que lleva 3 meses en la Luna.
Escribe una carta a tu familia. Cuentales como te sientes, que has descubierto,
y cuanto los extranas. Firma la carta con tu nombre inventado."""

# Perfil 1: Informe tecnico
print("PERFIL: INFORME TECNICO")
print()
informe = generar_carta(prompt_carta_final, opciones={
    "temperature": 0.2,
    "top_p": 0.3,
    "num_predict": 150,
    "repeat_penalty": 1.0
})

print("\n" + "=" * 60 + "\n")

# Perfil 2: Carta personal
print("PERFIL: CARTA PERSONAL")
print()
carta_personal = generar_carta(prompt_carta_final, opciones={
    "temperature": 0.8,
    "top_p": 0.9,
    "num_predict": 250,
    "repeat_penalty": 1.3
})

PERFIL: INFORME TECNICO

Parametros usados: {'temperature': 0.2, 'top_p': 0.3, 'num_predict': 150, 'repeat_penalty': 1.0}
##  Carta desde la Luna

Queridos seres queridos,

Escribir esta carta es un pequeño ritual que me llena de nostalgia y esperanza.  Llevo ya tres meses en la Luna, y aunque el silencio y la soledad son constantes, también he descubierto una paz inigualable. 

El paisaje lunar es una belleza que no se puede describir con palabras. Las montañas de roca, las llanuras de polvo y los cráteres, todo está tan ordenado y tan misterioso a la vez.  He aprendido a apreciar la quietud del espacio, la inmensidad del universo y la fragilidad de la vida. 

He dedicado mucho tiempo a la investigación, estudiando la composición de la roca lunar y la influencia de la gravedad

[Tokens generados: 150]


PERFIL: CARTA PERSONAL

Parametros usados: {'temperature': 0.8, 'top_p': 0.9, 'num_predict': 250, 'repeat_penalty': 1.3}
## Mis Queridas Familias,

Lo escribo de nuevo desde este peque

## 8. Misión para casa: Tu propia carta lunar

Ahora es tu turno. Tu tarea es:

### Ejercicio

1. **Crea tu propio prompt** con un escenario lunar (puede ser: describir un cráter, contar un problema técnico, narrar un descubrimiento, etc.)
2. **Experimenta con al menos 3 combinaciones diferentes** de parámetros tpo_p temperatura penalizacion y numero maximo
3. **Registra tus observaciones**: para cada combinación, anota qué cambió y por qué crees que pasó

In [14]:
# Tu prompt personalizado
mi_prompt = """Eres un astronauta en la Luna. Acabas de descubrir una base nodriza con muchas naves
cerca de la base. Escribe un parrafo contandole a tu familia sobre este descubrimiento y como podria cambiar la vida de todo el planeta."""

# Combinacion 1: Creativa y extensa
print("COMBINACION 1: Creativa y extensa")
print()
resultado_1 = generar_carta(mi_prompt, opciones={
    "temperature": 1.0,
    "top_p": 0.9,
    "num_predict": 200,
    "repeat_penalty": 1.3
})

COMBINACION 1: Creativa y extensa

Parametros usados: {'temperature': 1.0, 'top_p': 0.9, 'num_predict': 200, 'repeat_penalty': 1.3}
¡Hola, familia!  No puedo creer lo que acabé encontrando... ¡una base nodriza completamente funcional en la Luna! Y no es una simple estación: hay cientos de naves ahí, todas con una tecnología impresionante que me deja sin palabras. Imagínate si podemos empezar a explotar el potencial del espacio para construir nuevas formas de vida aquí abajo -  construir ciudades espaciales o desarrollar energía limpia y abundante gracias al sol lunar. Podría revolucionar la forma en que vive cada ser humano; es un amanecer para nosotros como humanidad, capaz de romper barreras tecnológicas y explorar nuevos horizontes. No tengo palabras. ¡Esperemos que esto sea lo que realmente necesitamos!  


[Tokens generados: 143]


In [15]:
# Combinacion 2: Precisa y corta (modo telegrama tecnico)
print("COMBINACION 2: Precisa y corta")
print()
resultado_2 = generar_carta(mi_prompt, opciones={
    "temperature": 0.2,
    "top_p": 0.3,
    "num_predict": 50,
    "repeat_penalty": 1.0
})

COMBINACION 2: Precisa y corta

Parametros usados: {'temperature': 0.2, 'top_p': 0.3, 'num_predict': 50, 'repeat_penalty': 1.0}
¡Hola a todos!  No puedo creer lo que acabé de descubrir.  Estoy en la Luna y justo al lado de una base nodriza con una cantidad asombrosa de naves.  Parece que se trata de una base de tecnología avanzada

[Tokens generados: 50]


In [16]:
# Combinacion 3: Equilibrada con vocabulario variado
print("COMBINACION 3: Equilibrada con vocabulario variado")
print()
resultado_3 = generar_carta(mi_prompt, opciones={
    "temperature": 0.6,
    "top_p": 0.85,
    "num_predict": 150,
    "repeat_penalty": 1.5
})

COMBINACION 3: Equilibrada con vocabulario variado

Parametros usados: {'temperature': 0.6, 'top_p': 0.85, 'num_predict': 150, 'repeat_penalty': 1.5}
¡Hola, familia!  No puedo creer lo que acabé encontrando... He llegado al punto más emocionante en mi viaje hasta ahora: una base nodriza con cientos de naves alrededor del borde lunar. Es impresionante; se extiende por kilómetros y parece estar diseñada para un propósito monumentalmente importante. Me imagino miles de personas trabajando aquí, impulsados ​​por la ambición humana que siempre ha caracterizado a nuestro planeta... ¡es como si el futuro mismo estuviera en nuestras manos!  Esta base podría cambiar las vidas del mundo entero: desde viajes espaciales más eficientes hasta una revolución tecnológica sin precedentes; incluso un nuevo paradigma para nuestra civilización. ¿Qué pasaría con los recursos, la ciencia y la tecnología? La Luna es ahora nuestro puente

[Tokens generados: 150]


### Preguntas guía

- ¿Cuál combinación produjo el texto más "humano"? la combinacion 3
- ¿Cuál fue la más útil para un reporte técnico? la combinacion 1
- ¿Qué pasa si subes la temperature a 2.0? ¿Y si la bajas a 0.0?
- ¿Cuál parámetro crees que tiene el mayor impacto en la calidad del texto? Top_P y temperature

### Bonus: Usando `ollama.chat()`

`ollama.chat()` permite usar mensajes con roles (system, user, assistant), similar a como funcionan ChatGPT o Claude. Prueba el siguiente ejemplo:

In [17]:
# Bonus: usando ollama.chat() con roles de conversacion
respuesta = ollama.chat(
    model="gemma2:2b",
    messages=[
        {"role": "system", "content": "Eres un astronauta poetico renacentista en la Luna."},
        {"role": "user", "content": "Describe el anochecer terrestre visto desde la Luna."}
    ],
    options={"temperature": 0.9, "top_p": 0.85, "num_predict": 200},
    think=False
)
print(respuesta["message"]["content"])

La quietud lunar me envuelve, un manto de polvo y silencio, mientras la esfera azul se despide del día. La Tierra, una joya en la negrura, cambia lentamente de color. 

Primero, un naranja vibrante, como si un pintor alquímico hubiera usado fuego para pintar su piel.  Las ciudades, diminutas luces que bailan sobre la tierra, se confunden con los faros de barcos a vela. Los ríos son serpientes de plata y oro, deslizándose entre las montañas doradas.

Luego, un rojo intenso, como el corazón de un toro furioso, se expande por la esfera terrestre. Las nubes, vestidas con fuego, se agolpan en una danza frenética, mientras la luz del sol se desvanece, dejando un resplandor tenue. 

El cielo, ahora un lienzo de negro salpicado por las últimas luces de la Tierra, me recuerda a los cielos de la pintura renacentista. La obra


---

**Fin de la transmisión. Base Selene, fuera.**

In [ ]:
Lyrken Calle